In [ ]:
import numpy as np

class UniversalSimplexSolver:
    def __init__(self, c, A, b, constraint_types):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.types = constraint_types  # list of '<=', '>=', '='
        self.m, self.n = self.A.shape
        self.tableau = None

    def solve(self):
        # 1. 右辺bが負の場合、正にする（制約を反転）
        for i in range(self.m):
            if self.b[i] < 0:
                self.A[i] *= -1
                self.b[i] *= -1
                if self.types[i] == '<=': self.types[i] = '>='
                elif self.types[i] == '>=': self.types[i] = '<='

        # 2. 標準形と人工変数の追加（2段階法準備）
        # 変数構成: 元の変数(n) + スラック(s) + 人工変数(a)
        num_slack = self.types.count('<=') + self.types.count('>=')
        num_artif = self.types.count('>=') + self.types.count('=')

        total_vars = self.n + num_slack + num_artif
        self.tableau = np.zeros((self.m + 2, total_vars + 1)) # +2は目的関数行と人工目的関数行

        slack_idx = self.n
        artif_idx = self.n + num_slack
        artif_rows = []

        for i in range(self.m):
            self.tableau[i+2, :self.n] = self.A[i]
            if self.types[i] == '<=':
                self.tableau[i+2, slack_idx] = 1
                slack_idx += 1
            elif self.types[i] == '>=':
                self.tableau[i+2, slack_idx] = -1
                self.tableau[i+2, artif_idx] = 1
                artif_rows.append(i+2)
                slack_idx += 1
                artif_idx += 1
            elif self.types[i] == '=':
                self.tableau[i+2, artif_idx] = 1
                artif_rows.append(i+2)
                artif_idx += 1
            self.tableau[i+2, -1] = self.b[i]

        # 第1段階の目的関数 (人工変数の和を最小化)
        for r in artif_rows:
            self.tableau[1] -= self.tableau[r]
        self.tableau[1, self.n + num_slack : total_vars] = 0 # 人工変数列を0に整地

        # 第2段階の目的関数 (元の最小化問題: 係数そのまま)
        self.tableau[0, :self.n] = self.c

        # --- Phase 1: 人工変数を排除 ---
        if num_artif > 0:
            self._optimize(phase=1)
            if abs(self.tableau[1, -1]) > 1e-9:
                return "実行可能解なし"

        # --- Phase 2: 最適化 ---
        # 人工変数列を削除
        self.tableau = np.delete(self.tableau, range(self.n + num_slack, total_vars), axis=1)
        self.tableau = np.delete(self.tableau, 1, axis=0) # 人工目的関数行を削除
        return self._optimize(phase=2)

    def _optimize(self, phase):
        while True:
            # Blandのルール: 負の係数のうち最小の添字を選択
            pivot_col = -1
            for j in range(self.tableau.shape[1] - 1):
                if self.tableau[0, j] < -1e-9: # 最小化問題での判定
                    pivot_col = j
                    break

            if pivot_col == -1: break # 最適

            # 比率テスト
            ratios = []
            for i in range(1, self.tableau.shape[0]):
                val = self.tableau[i, pivot_col]
                if val > 1e-9:
                    ratios.append(self.tableau[i, -1] / val)
                else:
                    ratios.append(np.inf)

            pivot_row = np.argmin(ratios) + 1
            if ratios[pivot_row-1] == np.inf:
                return "非有界"

            # ピボット操作
            self.tableau[pivot_row, :] /= self.tableau[pivot_row, pivot_col]
            for i in range(self.tableau.shape[0]):
                if i != pivot_row:
                    self.tableau[i, :] -= self.tableau[i, pivot_col] * self.tableau[pivot_row, :]

        return self.tableau[0, -1], self.tableau[1:, -1]

def generate_random_problem():
    """問題をランダムに生成する"""
    n = np.random.randint(2, 6)  # 変数 2~5
    m = np.random.randint(2, 6)  # 制約 2~5

    c = np.random.uniform(-10, 10, size=n)
    A = np.random.uniform(-5, 15, size=(m, n))
    b = np.random.uniform(5, 50, size=m)
    types = np.random.choice(['<=', '>=', '='], size=m)

    print(f"--- ランダム生成問題 (変数:{n}, 制約:{m}) ---")
    print(f"目的関数係数 c: {c.round(2)}")
    print(f"制約タイプ: {types}")
    return c, A, b, types

# 実行
c, A, b, types = generate_random_problem()
solver = UniversalSimplexSolver(c, A, b, types)
result = solver.solve()

if isinstance(result, tuple):
    val, sol = result
    print(f"最適値: {val:.4f}")
else:
    print(f"結果: {result}")